In [1]:
# default_exp meta_trainer

%load_ext autoreload
%autoreload 2

# Meta optimizer

> API details.

In [2]:
#export
from enum import Enum
class OptimizerType(Enum):
    GUROBI = 1
    OR_TOOLS = 2

In [22]:
#export
import tensorflow as tf
from tensorflow.keras import models 
from tensorflow.keras import layers
from tensorflow.keras.callbacks import ModelCheckpoint
import gurobipy as gp
from gurobipy import GRB
import numpy as np
from ortools.linear_solver import pywraplp
import pandas as pd
import numpy as np

import random
import tensorflow as tf
from tensorflow.keras import models
from tensorflow.keras import layers
from tensorflow.keras.callbacks import ModelCheckpoint

class MetaOptimizer:


    def __init__(self,optimizer_type : OptimizerType):
        self.num_hidden_nodes = 1000
        self.optimizer_type = optimizer_type
        return

    def train_model(self, x_training, y_training, X_validation, y_validation):
        self.num_input_cols = x_training.shape[1]
        self.num_ouptup_cols = y_training.shape[1]

        NN_model = models.Sequential()
        NN_model.add(layers.Dense(self.num_hidden_nodes,activation='relu'))  #One hidden layer with 1000 nodes
        NN_model.add(layers.Dense(self.num_ouptup_cols))
        NN_model.compile(optimizer='adam', loss='mean_absolute_error')
        chk = ModelCheckpoint('ANN_model',monitor='val_loss',save_best_only=True,mode='min',verbose=1)
        self.history = NN_model.fit(x_training,y_training,epochs=100,callbacks=[chk],validation_data=(X_validation,y_validation))

        self.NN_model = tf.keras.models.load_model('ANN_model')

        self.weights_1 = np.array(self.NN_model.get_weights()[0])
        self.weights_2 = np.array(self.NN_model.get_weights()[1])
        self.weights_3 = np.array(self.NN_model.get_weights()[2])
        self.weights_4 = np.array(self.NN_model.get_weights()[3])

    def solve_optimization(self, target_val, lower_bound, upper_bound):
        self.lower_bound = lower_bound
        self.upper_bound = upper_bound
        if self.optimizer_type == OptimizerType.GUROBI :
            return self._solve_optimization_gurobi(target_val)
        else :
            return self._solve_optimization_or_tools(target_val)

        
    def _solve_optimization_gurobi(self, target_val):
        # Create initial model
        model = gp.Model('project')
        model.Params.LogToConsole = 0
        model.Params.MIPFocus = 1
        model.setParam('MIPGap', 0.000001)

        #Index

        I = self.num_input_cols
        J = self.num_ouptup_cols
        L = self.num_hidden_nodes

        #Parameters

        M = self.num_hidden_nodes
        h = target_val
        w1 = self.weights_1
        b1 = self.weights_2
        w2 = self.weights_3
        b2 = self.weights_4

        #Decision Variables

        x = model.addVars(I,vtype=GRB.CONTINUOUS,name='x',lb=self.lower_bound,ub=self.upper_bound)  #the value of parameter i
        y = model.addVars(L, vtype=GRB.CONTINUOUS,name='y')  #the hidden node value in output j for node l
        z = model.addVars(J, vtype=GRB.CONTINUOUS,name='z')  #the model value for output j
        u = model.addVars(L, vtype=GRB.BINARY,name='u')  #the binary indicator for activation of l th hidden node
        d = model.addVars(J, vtype=GRB.CONTINUOUS,name='d')  #the difference value for output j

        # Constraints:

        model.addConstrs((gp.quicksum(w1[i, l] * x[i] for i in range(I)) + b1[l] <= y[l] for l in range(L)),name='FirstConsts')
        model.addConstrs((gp.quicksum(w1[i, l] * x[i] for i in range(I)) + b1[l] + M * (1 - u[l]) >= y[l] for l in range(L)),name='SecondConsts')
        model.addConstrs((y[l] <= M * u[l] for l in range(L)),name='ThirdCOnsts')
        model.addConstrs((gp.quicksum(y[l] * w2[l, j] for l in range(L)) + b2[j] == z[j]for j in range(J)),name='FourthConsts')
        model.addConstrs((d[j] >= z[j] - h[j] for j in range(J)),name='FifthConsts')
        model.addConstrs((d[j] >= h[j] - z[j] for j in range(J)),name='SixthConsts')

        # Set global sense for ALL objectives
        model.ModelSense = GRB.MINIMIZE

        obj = gp.quicksum(d[j] for j in range(J))
        model.setObjective(obj)

        # Optimize

        model.optimize()

        parameter_list = []
        for i in range(I):
            parameter_list.append(x[i].x)

        output_list = []
        for i in range(J):
            output_list.append(z[i].x)
        """
        for v in model.getVars():
            print('%s %g' % (v.varName, v.x))
                
        print('Obj: %g' % obj.getValue())
        """

        return parameter_list, output_list
    
    def _solve_optimization_or_tools(self, target_val):
        solver = pywraplp.Solver.CreateSolver('SCIP')

        mip_gap = 0.000001

        #model = pywraplp.Solver('model', pywraplp.Solver.SCIP_MIXED_INTEGER_PROGRAMMING)

        solverParams = pywraplp.MPSolverParameters()
        solverParams.SetDoubleParam(solverParams.RELATIVE_MIP_GAP, mip_gap)

        #Index

        I = self.num_input_cols
        J = self.num_ouptup_cols
        L = self.num_hidden_nodes

        #Parameters

        M = self.num_hidden_nodes
        h = target_val
        w1 = self.weights_1
        b1 = self.weights_2
        w2 = self.weights_3
        b2 = self.weights_4

        lb = [1.0, 0.5, 5.0, 1.0, 0.4, 390.0]
        ub = [2.5, 1.0, 10.0, 5.0, 0.8, 760.0]
        infinity = solver.infinity()
        x_vars = {}
        y_vars = {}
        z_vars = {}
        u_vars = {}
        d_vars = {}

        for i in range(I):
            x_vars[i] = solver.NumVar(lb=self.lower_bound[i], ub=self.upper_bound[i], name=f'x[{i}]')
        for i in range(L):
            y_vars[i] = solver.NumVar(lb=0.0, ub=infinity, name=f'y[{i}]')
        for i in range(J):
            z_vars[i] = solver.NumVar(lb=0.0, ub=infinity, name=f'z[{i}]')
        for i in range(L):
            u_vars[i] = solver.BoolVar(name=f'u[{i}]')
        for i in range(J):
            d_vars[i] = solver.NumVar(lb=0.0, ub=infinity, name=f'd[{i}]')

        for l in range(L):
            constraint_expr = [w1[i,l] * x_vars[i] for i in range(I)]
            solver.Add(sum(constraint_expr) + b1[l] <= y_vars[l])

        for l in range(L):
            constraint_expr = [w1[i,l] * x_vars[i] for i in range(I)]
            solver.Add(sum(constraint_expr) + b1[l] + M * (1 - u_vars[l]) >= y_vars[l])

        for l in range(L):
            solver.Add(M * u_vars[l] >= y_vars[l])

        for j in range(J):
            constraint_expr = [w2[l,j] * y_vars[l] for l in range(L)]
            solver.Add(sum(constraint_expr) + b2[j] == z_vars[j])

        for j in range(J):
            solver.Add(z_vars[j] - h[j] <= d_vars[j])

        for j in range(J):
            solver.Add(h[j] - z_vars[j] <= d_vars[j])


        obj_expr = [d_vars[j] for j in range(J)]
        solver.Minimize(solver.Sum(obj_expr))

        status = solver.Solve()

        self.parameter_list = [x_vars[i].solution_value() for i in range(I)]


        self.output_list = [z_vars[j].solution_value() for j in range(J)]

        return self.parameter_list, self.output_list


## Testing

In [ ]:
M = pd.read_csv('output-seed_1337-30000.csv')
#NORMALIZATION
M.drop(M.columns[[8,12,13,16,17,18,21]], axis = 1, inplace= True)
for i in range(6,15):
    M.iloc[:,i] = (M.iloc[:,i] - M.iloc[:,i].min())/(M.iloc[:,i].max() - M.iloc[:,i].min())
    target_sample = M.sample(100)
M = M.drop(target_sample.index)
M.reset_index(inplace=True)
M.drop(M.columns[0], axis = 1, inplace=True)
M.to_csv('OutputMinMaxScaledResults30000.csv')
target_sample.to_csv('NewTargetSample.csv')

M_train = M.sample(frac=0.7,  random_state=1337)
M_rest = M.drop(M_train.index)
M_validation = M_rest.sample(frac=0.5,  random_state=1337)
M_test = M_rest.drop(M_validation.index)

X_training = np.array(M_train.iloc[:,:6])
X_validation = np.array(M_validation.iloc[:,:6])
X_test = np.array(M_test.iloc[:,:6])

Y_training = np.array(M_train.iloc[:,6:])
Y_validation = np.array(M_validation.iloc[:,6:])
Y_test = np.array(M_test.iloc[:,6:])

In [13]:
Target_Sample = pd.read_csv('NewTargetSample.csv', index_col=0).iloc[:,[6,7,8,9,10,11,12,13,14]]
Target_Sample.reset_index(inplace=True)
Target_Sample.drop(Target_Sample.columns[0], axis = 1, inplace=True)
target_val = np.array(Target_Sample[0:1]).tolist()[0]

In [ ]:
optimizer = MetaOptimizer(OptimizerType.OR_TOOLS)
optimizer.train_model(X_training, Y_training, X_validation, Y_validation)

In [ ]:
sample_arrays = np.array(Target_Sample)
lower_bound = [1, 0.5, 5, 1, 0.4, 390]
upper_bound = [2.5, 1, 10, 5, 0.8,760]

results_param = {}
results_output = {}
for i in range(100):
    result = optimizer.solve_optimization(sample_arrays[i],lower_bound,upper_bound)
    results_param[i] = result[0]
    results_output[i] = result[1]
    print(f'{i} done')


In [ ]:
print(results_param)  #GUROBI

In [ ]:
print(results_output)  #GUROBI

In [ ]:
print(results_param)  #OR_TOOLS

In [ ]:
print(results_output)  #OR_TOOLS